# Lab 3: 语音合成 — Qwen3-TTS + OpenVINO

Qwen3-TTS 是阿里巴巴通义千问团队推出的最先进文本转语音模型。它支持多语言语音合成、声音克隆和声音设计等功能。

**主要特点:**
- 🌍 覆盖中、英、日、韩、德、法、俄、葡、西、意 10 大语言
- 🎭 支持多种预设说话人和风格指令控制
- ⚡ 基于双轨混合流式生成架构，首包延迟低至 97ms
- 🧠 智能文本理解，自适应调整语调、语速和情感表达

在本实验中，我们将：
1. 从 ModelScope 下载预转换的 OpenVINO 模型
2. 使用 OpenVINO 加载并运行语音合成
3. 体验交互式 Gradio 语音合成演示

#### 目录：
- [环境准备](#环境准备)
- [下载模型](#下载模型)
- [选择推理设备](#选择推理设备)
- [加载模型](#加载模型)
- [运行语音合成](#运行语音合成)
- [交互式演示](#交互式演示)

## 环境准备
[返回目录 ⬆️](#目录：)

### Step 1：创建 Python 虚拟环境

```bash
python -m venv .venv
```

激活虚拟环境：
```bash
.venv/Scripts/activate
```

### Step 2：安装依赖

 > ⚠️ 注意：请确保 OpenVINO 和 OpenVINO GenAI 版本均不低于 2026.1。

```bash
pip install --upgrade pip
pip install modelscope gradio ipywidgets IPython torch transformers
pip install openvino>=2026.1 openvino-genai>=2026.1
```

### Step 3：检查 helper 文件

`from qwen_3_tts_helper import OVQwen3TTSModel` 依赖当前目录存在 `qwen_3_tts_helper.py`。

请确保 notebook 在 `lab3-text-to-speech/` 目录下运行，并检查文件是否存在：

```bash
dir qwen_3_tts_helper.py
```

## 下载模型
[返回目录 ⬆️](#目录：)

从 ModelScope 下载已转换的 Qwen3-TTS-CustomVoice-0.6B OpenVINO 模型。如果模型已存在则跳过下载。

In [ ]:
from pathlib import Path

model_dir = Path("Qwen3-TTS-CustomVoice-0.6B-fp16-ov")

if not model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("snake7gun/Qwen3-TTS-CustomVoice-0.6B-fp16-ov", local_dir=str(model_dir))
    print(f"模型已下载到: {model_dir}")
else:
    print(f"模型已存在: {model_dir}，跳过下载")

## 选择推理设备
[返回目录 ⬆️](#目录：)

选择用于推理的设备。默认使用GPU，无法使用再退回CPU。

In [ ]:
from notebook_utils import device_widget

device = device_widget("GPU", exclude=["NPU"])
device

## 加载模型
[返回目录 ⬆️](#目录：)

使用 `OVQwen3TTSModel` 加载 OpenVINO 模型。该 wrapper 提供与原始模型相同的 API 接口。

In [ ]:
from qwen_3_tts_helper import OVQwen3TTSModel

ov_model = OVQwen3TTSModel.from_pretrained(
    model_dir=model_dir,
    device=device.value,
)

print(f"\n✅ 模型加载完成!")
print(f"   模型类型: {ov_model.tts_model_type}")
print(f"   模型大小: {ov_model.tts_model_size}")
print(f"   支持的说话人: {ov_model.get_supported_speakers()}")
print(f"   支持的语言: {ov_model.get_supported_languages()}")

## 运行语音合成
[返回目录 ⬆️](#目录：)

测试 CustomVoice 模式的语音合成。你可以指定不同的说话人和风格指令。

In [ ]:
import time
import IPython.display as ipd

test_text = "你好！欢迎来到 OpenVINO Workshop，这是一个语音合成的测试。"

print(f"输入文本: {test_text}")
print("\n正在生成语音...")

start_time = time.time()

wavs, sr = ov_model.generate_custom_voice(
    text=test_text,
    speaker="vivian",
    language="Chinese",
    instruct="用友好亲切的语气说话。",
    non_streaming_mode=True,
    max_new_tokens=2048,
)

inference_time = time.time() - start_time

if wavs is not None:
    audio_duration = len(wavs[0]) / sr
    print(f"\n✅ 语音生成完成!")
    print(f"   推理时间: {inference_time:.2f}s")
    print(f"   音频时长: {audio_duration:.2f}s")
    print(f"   实时因子 (RTF): {inference_time/audio_duration:.3f}")
    print(f"   采样率: {sr} Hz")

    ipd.display(ipd.Audio(wavs[0], rate=sr))

## 交互式演示
[返回目录 ⬆️](#目录：)

启动 Gradio 交互式演示，你可以：
- 输入任意文本进行语音合成
- 选择不同的说话人
- 选择语言
- 添加风格指令

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, model_type=ov_model.tts_model_type)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)